# ETL do Dataset Olist

Este notebook executa a etapa de ETL dos CSVs da pasta `dataset`.

Nesta fase, o objetivo e limpar, padronizar, transformar e integrar os dados brutos. A modelagem de fatos e dimensoes fica para uma etapa futura.

## 1. Importacao de bibliotecas e configuracoes

In [ ]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_DIR = BASE_DIR / 'dataset'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

DATASET_DIR, OUTPUT_DIR

## 2. Funcoes auxiliares

In [ ]:
UFS_BRASIL = {
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG',
    'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
}

def remover_acentos(valor):
    if pd.isna(valor):
        return valor
    texto = str(valor)
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(char for char in texto if not unicodedata.combining(char))

def padronizar_texto(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.lower()
        .map(remover_acentos)
        .astype('string')
    )

def padronizar_uf(serie):
    uf = serie.astype('string').str.strip().str.upper()
    return uf.where(uf.isin(UFS_BRASIL), pd.NA)

def padronizar_cep(serie):
    return serie.astype('string').str.extract(r'(\d+)', expand=False).str.zfill(5)

def converter_datas(df, colunas):
    for coluna in colunas:
        df[coluna] = pd.to_datetime(df[coluna], errors='coerce')
    return df

def remover_duplicados(df, nome):
    antes = len(df)
    df = df.drop_duplicates().copy()
    depois = len(df)
    print(f'{nome}: {antes - depois} duplicados removidos')
    return df

def resumo_tabela(nome, df):
    return {
        'tabela': nome,
        'linhas': len(df),
        'colunas': df.shape[1],
        'duplicados': int(df.duplicated().sum()),
        'nulos_total': int(df.isna().sum().sum())
    }

## 3. Extracao dos CSVs brutos

In [ ]:
arquivos = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

dfs = {
    nome: pd.read_csv(DATASET_DIR / arquivo)
    for nome, arquivo in arquivos.items()
}

for nome, df in dfs.items():
    print(f'{nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

## 4. Perfilamento inicial

In [ ]:
perfil_inicial = pd.DataFrame([
    resumo_tabela(nome, df) for nome, df in dfs.items()
])
perfil_inicial

In [ ]:
for nome, df in dfs.items():
    print('\n' + '=' * 80)
    print(nome)
    display(df.head(3))
    display(df.isna().sum().sort_values(ascending=False).head(10))

## 5. Transformacao e limpeza das tabelas

In [ ]:
customers = remover_duplicados(dfs['customers'], 'customers')
customers['customer_zip_code_prefix'] = padronizar_cep(customers['customer_zip_code_prefix'])
customers['customer_city'] = padronizar_texto(customers['customer_city'])
customers['customer_state'] = padronizar_uf(customers['customer_state'])

sellers = remover_duplicados(dfs['sellers'], 'sellers')
sellers['seller_zip_code_prefix'] = padronizar_cep(sellers['seller_zip_code_prefix'])
sellers['seller_city'] = padronizar_texto(sellers['seller_city'])
sellers['seller_state'] = padronizar_uf(sellers['seller_state'])

In [ ]:
orders = remover_duplicados(dfs['orders'], 'orders')
orders['order_status'] = padronizar_texto(orders['order_status'])
orders = converter_datas(orders, [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

orders['ano_pedido'] = orders['order_purchase_timestamp'].dt.year
orders['mes_pedido'] = orders['order_purchase_timestamp'].dt.month
orders['trimestre_pedido'] = orders['order_purchase_timestamp'].dt.quarter
orders['dia_semana_pedido'] = orders['order_purchase_timestamp'].dt.day_name()

orders['dias_entrega'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days
orders['dias_estimados_entrega'] = (
    orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']
).dt.days
orders['atraso_dias'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days
orders['atraso_dias'] = orders['atraso_dias'].clip(lower=0)
orders['entregue_no_prazo'] = np.where(orders['atraso_dias'].fillna(0) == 0, 1, 0)

In [ ]:
order_items = remover_duplicados(dfs['order_items'], 'order_items')
order_items = converter_datas(order_items, ['shipping_limit_date'])

for coluna in ['price', 'freight_value']:
    order_items[coluna] = pd.to_numeric(order_items[coluna], errors='coerce')
    order_items.loc[order_items[coluna] < 0, coluna] = np.nan
    order_items[coluna] = order_items[coluna].fillna(0)

order_items['valor_total_item'] = order_items['price'] + order_items['freight_value']

In [ ]:
products = remover_duplicados(dfs['products'], 'products')
translation = remover_duplicados(dfs['category_translation'], 'category_translation')

products['product_category_name'] = padronizar_texto(products['product_category_name']).fillna('sem_categoria')
translation['product_category_name'] = padronizar_texto(translation['product_category_name'])
translation['product_category_name_english'] = padronizar_texto(translation['product_category_name_english'])

products = products.merge(translation, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna(products['product_category_name'])

colunas_numericas_produto = [
    'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'
]

for coluna in colunas_numericas_produto:
    products[coluna] = pd.to_numeric(products[coluna], errors='coerce')
    products.loc[products[coluna] < 0, coluna] = np.nan
    products[coluna] = products[coluna].fillna(products[coluna].median())

In [ ]:
payments = remover_duplicados(dfs['payments'], 'payments')
payments['payment_type'] = padronizar_texto(payments['payment_type'])

payments['payment_installments'] = pd.to_numeric(payments['payment_installments'], errors='coerce').fillna(0)
payments['payment_value'] = pd.to_numeric(payments['payment_value'], errors='coerce').fillna(0)
payments.loc[payments['payment_installments'] < 0, 'payment_installments'] = 0
payments.loc[payments['payment_value'] < 0, 'payment_value'] = 0

payments_agregado = (
    payments.groupby('order_id', as_index=False)
    .agg(
        valor_total_pagamento=('payment_value', 'sum'),
        quantidade_pagamentos=('payment_sequential', 'count'),
        max_parcelas=('payment_installments', 'max'),
        quantidade_formas_pagamento=('payment_type', 'nunique'),
        forma_pagamento_principal=('payment_type', lambda s: s.mode().iat[0] if not s.mode().empty else 'indefinido')
    )
)

In [ ]:
reviews = remover_duplicados(dfs['reviews'], 'reviews')
reviews = converter_datas(reviews, ['review_creation_date', 'review_answer_timestamp'])
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('').astype('string').str.strip()
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('').astype('string').str.strip()
reviews['review_score'] = pd.to_numeric(reviews['review_score'], errors='coerce')
reviews.loc[~reviews['review_score'].between(1, 5), 'review_score'] = np.nan

reviews['tem_comentario'] = np.where(reviews['review_comment_message'].str.len() > 0, 1, 0)

reviews_agregado = (
    reviews.groupby('order_id', as_index=False)
    .agg(
        review_score_medio=('review_score', 'mean'),
        quantidade_reviews=('review_id', 'count'),
        tem_comentario=('tem_comentario', 'max'),
        primeira_review=('review_creation_date', 'min'),
        ultima_resposta_review=('review_answer_timestamp', 'max')
    )
)

In [ ]:
geolocation = remover_duplicados(dfs['geolocation'], 'geolocation')
geolocation['geolocation_zip_code_prefix'] = padronizar_cep(geolocation['geolocation_zip_code_prefix'])
geolocation['geolocation_city'] = padronizar_texto(geolocation['geolocation_city'])
geolocation['geolocation_state'] = padronizar_uf(geolocation['geolocation_state'])

geolocation['geolocation_lat'] = pd.to_numeric(geolocation['geolocation_lat'], errors='coerce')
geolocation['geolocation_lng'] = pd.to_numeric(geolocation['geolocation_lng'], errors='coerce')

geolocation_agregado = (
    geolocation.groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean'),
        geolocation_city=('geolocation_city', lambda s: s.mode().iat[0] if not s.mode().empty else pd.NA),
        geolocation_state=('geolocation_state', lambda s: s.mode().iat[0] if not s.mode().empty else pd.NA)
    )
)

## 6. Integracao dos dados tratados

In [ ]:
geo_cliente = geolocation_agregado.add_prefix('customer_')
geo_vendedor = geolocation_agregado.add_prefix('seller_')

pedidos_itens = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(order_items, on='order_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
    .merge(payments_agregado, on='order_id', how='left')
    .merge(reviews_agregado, on='order_id', how='left')
    .merge(geo_cliente, left_on='customer_zip_code_prefix', right_on='customer_geolocation_zip_code_prefix', how='left')
    .merge(geo_vendedor, left_on='seller_zip_code_prefix', right_on='seller_geolocation_zip_code_prefix', how='left')
)

pedidos_itens['valor_total_pagamento'] = pedidos_itens['valor_total_pagamento'].fillna(0)
pedidos_itens['quantidade_pagamentos'] = pedidos_itens['quantidade_pagamentos'].fillna(0)
pedidos_itens['quantidade_formas_pagamento'] = pedidos_itens['quantidade_formas_pagamento'].fillna(0)
pedidos_itens['forma_pagamento_principal'] = pedidos_itens['forma_pagamento_principal'].fillna('sem_pagamento')
pedidos_itens['review_score_medio'] = pedidos_itens['review_score_medio'].fillna(0)
pedidos_itens['quantidade_reviews'] = pedidos_itens['quantidade_reviews'].fillna(0)
pedidos_itens['tem_comentario'] = pedidos_itens['tem_comentario'].fillna(0)

pedidos_itens.shape

## 7. Validacoes finais

In [ ]:
tabelas_tratadas = {
    'customers_tratado': customers,
    'orders_tratado': orders,
    'order_items_tratado': order_items,
    'products_tratado': products,
    'sellers_tratado': sellers,
    'payments_agregado': payments_agregado,
    'reviews_agregado': reviews_agregado,
    'geolocation_agregado': geolocation_agregado,
    'pedidos_itens_tratado': pedidos_itens
}

relatorio_qualidade = pd.DataFrame([
    resumo_tabela(nome, df) for nome, df in tabelas_tratadas.items()
])
relatorio_qualidade

In [ ]:
validacoes = {
    'pedidos_sem_cliente': int(pedidos_itens['customer_id'].isna().sum()),
    'itens_sem_produto': int(pedidos_itens['product_id'].isna().sum()),
    'precos_negativos': int((pedidos_itens['price'].fillna(0) < 0).sum()),
    'fretes_negativos': int((pedidos_itens['freight_value'].fillna(0) < 0).sum()),
    'ufs_clientes_invalidas': int(customers['customer_state'].isna().sum()),
    'ufs_vendedores_invalidas': int(sellers['seller_state'].isna().sum())
}

pd.DataFrame(list(validacoes.items()), columns=['validacao', 'valor'])

## 8. Gravacao dos arquivos tratados

In [ ]:
for nome, df in tabelas_tratadas.items():
    caminho = OUTPUT_DIR / f'{nome}.csv'
    df.to_csv(caminho, index=False)
    print(f'Arquivo gerado: {caminho}')

relatorio_qualidade.to_csv(OUTPUT_DIR / 'relatorio_qualidade_etl.csv', index=False)
print('ETL concluido com sucesso.')

## 9. Analises exploratorias simples apos o ETL

Estas analises nao substituem a modelagem dimensional. Elas servem apenas para validar que a base tratada ja permite perguntas gerenciais.

In [ ]:
faturamento_mes = (
    pedidos_itens.groupby(['ano_pedido', 'mes_pedido'], as_index=False)
    .agg(faturamento=('valor_total_item', 'sum'), pedidos=('order_id', 'nunique'))
    .sort_values(['ano_pedido', 'mes_pedido'])
)
faturamento_mes.head(12)

In [ ]:
categorias_mais_vendidas = (
    pedidos_itens.groupby('product_category_name_english', as_index=False)
    .agg(quantidade_itens=('order_item_id', 'count'), receita=('valor_total_item', 'sum'))
    .sort_values('receita', ascending=False)
    .head(10)
)
categorias_mais_vendidas

In [ ]:
entrega_por_estado = (
    pedidos_itens.groupby('customer_state', as_index=False)
    .agg(
        pedidos=('order_id', 'nunique'),
        dias_entrega_medio=('dias_entrega', 'mean'),
        atraso_medio=('atraso_dias', 'mean'),
        avaliacao_media=('review_score_medio', 'mean')
    )
    .sort_values('pedidos', ascending=False)
)
entrega_por_estado.head(10)